# Classification from cytological measurements

This notebook accompanies Exercise 5.5 and brings together the chapter's ideas. The *Wisconsin Diagnostic Breast Cancer* dataset contains $569$ observations described by $30$ real-valued measurements extracted from cytological images.

We compare an affine SVM, a Gaussian SVM and an MLP. The aim is to study a dataset and the mathematical choices required for that study, rather than to validate a medical diagnostic device.

## Outline

1. [Data, labels and frequencies](#donnees-wisconsin)
2. [Splitting and standardisation](#pretraitement-wisconsin)
3. [Projection onto two principal components](#pca-wisconsin)
4. [Affine SVM and Gaussian SVM](#svm-wisconsin)
5. [MLP with Flax NNX](#mlp-wisconsin)
6. [Errors and signed margins](#erreurs-wisconsin)
7. [Asymmetric errors](#poids-wisconsin)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import flax
from flax import nnx
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

<a id="donnees-wisconsin"></a>
## 1. Data, labels and frequencies

Load the dataset with `load_breast_cancer`. In `scikit-learn`, target `0` denotes a malignant tumour and target `1` a benign tumour. Adopt the course convention
$$
z=+1\quad\text{malignant},\qquad z=-1\quad\text{benign}.
$$

Check the dimensions, display the names of the first measurements and compute the frequencies of both classes.

In [ ]:
# Complete this section.

<a id="pretraitement-wisconsin"></a>
## 2. Splitting and standardisation

Construct a fixed stratified split with approximately $60\%$ training data, $20\%$ validation data and $20\%$ test data.

Compare the orders of magnitude of the measurements, then fit `StandardScaler` **only** to the training data. Apply the same transformation to the other two sets. Using validation or test data in this calculation would constitute data leakage.

In [ ]:
# Complete this section.

<a id="pca-wisconsin"></a>
## 3. Projection onto two principal components

Compute the first two principal components using only the standardised training data. Project all three sets onto the resulting plane.

Separation in this plane would provide an affine score also separating the data in $\mathbb R^{30}$. However, overlap in the projected plane does not imply that the data are non-separable in the original space.

In [ ]:
# Complete this section.

<a id="svm-wisconsin"></a>
## 4. Affine SVM and Gaussian SVM

We use the `SVC` class from `scikit-learn`, whose implementation is based on LIBSVM.

For each machine, choose hyperparameters using the validation set:

- $C$ for the affine SVM;
- $C$ and $\gamma$ for the Gaussian SVM.

The `C` parameter of `SVC` is inversely related to regularisation strength. If validation frequencies are equal, use mean hinge loss as a secondary criterion.

In [ ]:
def perte_charniere(scores, z):
    return np.mean(np.maximum(0.0, 1.0 - z * scores))

def selectionner_svm(noyau, valeurs_C, valeurs_gamma=(None,)):
    candidats = []
    for C in valeurs_C:
        for gamma in valeurs_gamma:
            arguments = {"kernel": noyau, "C": C}
            if gamma is not None:
                arguments["gamma"] = gamma
            modele = SVC(**arguments)
            modele.fit(x_app_n, z_app)
            scores = modele.decision_function(x_val_n)
            erreur = np.mean(np.sign(scores) != z_val)
            candidats.append((erreur, perte_charniere(scores, z_val), C, gamma, modele))
    return min(candidats, key=lambda resultat: (resultat[0], resultat[1]))

In [ ]:
# Complete this section.

<a id="mlp-wisconsin"></a>
## 5. MLP with Flax NNX

Fit an MLP with two hidden layers using regularised logistic loss. Compare several widths, regularisation coefficients and initialisations. Select the machine using error frequency followed by logistic loss on the validation set.

The training functions are provided; the exercise concerns constructing and interpreting the comparison.

In [ ]:
class MLPBinaire(nnx.Module):
    def __init__(self, dimension, largeur, *, rngs):
        self.affine_1 = nnx.Linear(dimension, largeur, rngs=rngs)
        self.affine_2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.affine_3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        y = jnp.tanh(self.affine_1(x))
        y = jnp.tanh(self.affine_2(y))
        return self.affine_3(y)[..., 0]

def norme_parametres(machine):
    return sum(
        jnp.sum(feuille**2)
        for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param))
    )

def cout_mlp(machine, x, z, lamb):
    return (
        jnp.mean(jax.nn.softplus(-z * machine(x)))
        + 0.5 * lamb * norme_parametres(machine)
    )

@nnx.jit
def pas_mlp(machine, optimiseur, x, z, lamb):
    fonction = lambda m: cout_mlp(m, x, z, lamb)
    valeur, gradient = nnx.value_and_grad(fonction)(machine)
    optimiseur.update(machine, gradient)
    return valeur

def entrainer_mlp(x, z, largeur, lamb, graine, *, iterations=1200):
    machine = MLPBinaire(x.shape[1], largeur, rngs=nnx.Rngs(graine))
    optimiseur = nnx.Optimizer(machine, optax.adam(5.0e-3), wrt=nnx.Param)
    x_jax = jnp.asarray(x)
    z_jax = jnp.asarray(z)
    for _ in range(iterations):
        valeur = pas_mlp(machine, optimiseur, x_jax, z_jax, lamb)
    return machine, float(valeur)

In [ ]:
# Complete this section.

<a id="erreurs-wisconsin"></a>
## 6. Errors and signed margins

Compare the three machines on the training, validation and test sets. On the test set, distinguish:

- false negatives: malignant tumours classified as benign;
- false positives: benign tumours classified as malignant.

Plot the signed margins. Their signs are comparable between machines, but their scales depend on score normalisation.

In [ ]:
# Complete this section.

In [ ]:
# Complete this section.

<a id="poids-wisconsin"></a>
## 7. Asymmetric errors

In an application, the two error types may have different importance. The `class_weight` parameter of `SVC` weights the hinge-loss terms.

Fit affine SVMs with ratios $\omega_{+1}/\omega_{-1}=1,2,5$. Observe changes in false negatives and false positives on the test data. These weights are not automatically inferred from the data: they express an application-specific choice.

In [ ]:
# Complete this section.

## Summary

This real dataset brings together the chapter's main choices: data representation, score, loss, regularisation, validation and error measurement. The kernel SVM and MLP may improve on an affine score, but comparing them is meaningful only with a common experimental protocol throughout.